# Bucketing a tick stream: `every=`, `count=`, and empty bars

There are two ways to cut a tick stream into bars, and they measure along
different axes.

`every=W` buckets along the **index**. Bar `n` is the half-open interval
`[origin + n*W, origin + (n+1)*W)`, so the index values decide membership. Bars
have equal width on the index but a variable number of ticks. Boundaries are
anchored at `origin` (default `0`), not at the first tick.

`count=N` buckets by **arrival order**. A bar closes every `N` events and never
consults the index values to place boundaries. Bars hold an equal number of
ticks but span a variable width of the index.

The difference shows up wherever activity is uneven. This notebook uses one
stream with a deliberate hole in its clock: `count=` walks straight across the
hole without noticing, while `every=` produces bars that contain no ticks at
all. Those empty bars are what `fill=` controls.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer.streams import resample

rng = np.random.default_rng(7)
n = 80

# Dense clock, one tick per unit, with a single 130-unit hole after tick 40.
gaps = np.ones(n)
gaps[40] = 130
timestamps = np.cumsum(gaps).astype(np.int64)

# Mid-price: a slow random walk around 100.
price = 100.0 + np.cumsum(rng.standard_normal(n) * 0.3)

print(f"{n} ticks spanning index {timestamps[0]}..{timestamps[-1]}")
print(f"hole: {timestamps[39]} -> {timestamps[40]}")

## Time bars: a fixed interval with `every=`

`every=40` closes a bar every 40 units of the index, on a grid anchored at
`origin=0`. Alongside the mean price we take `agg="count"` to see how many ticks
landed in each bar. The dense stretches pack 40 ticks into a bar; across the
hole, bars thin out.

By default (`fill="skip"`) a bucket with no ticks at all emits no row, so the
bar labels below jump straight over the hole.

In [ ]:
EVERY = 40

time_mean  = resample(price, timestamps, every=EVERY, agg="mean")
time_count = resample(price, timestamps, every=EVERY, agg="count")

pd.DataFrame({
    "mean":    time_mean.values.round(3),
    "n_ticks": time_count.values.astype(int),
}, index=pd.Index(time_mean.index, name="bar_open"))

## Tick bars: a fixed event count with `count=`

`count=12` closes a bar every 12 ticks, regardless of how much index passes. The
tick count is now constant (the trailing bar may hold fewer), and it is the index
*width* that varies.

Under `count=`, a bar is labelled by an actual tick index: the first tick of the
bar for `label="left"` (the default), the last tick for `label="right"`. Taking
both gives the true span of each bar. Note that this is not the gap between
consecutive bar opens, which would also include the dead index between one bar's
last tick and the next bar's first.

One bar straddles the hole: it reaches across 130 units of empty index because
`count=` measures rows, not distance.

In [ ]:
COUNT = 12

tick_mean  = resample(price, timestamps, count=COUNT, agg="mean")
tick_count = resample(price, timestamps, count=COUNT, agg="count")

# label="left" gives each bar's first tick, label="right" its last tick.
first_tick = tick_mean.index
last_tick  = resample(price, timestamps, count=COUNT, agg="mean", label="right").index

pd.DataFrame({
    "mean":       tick_mean.values.round(3),
    "n_ticks":    tick_count.values.astype(int),
    "last_tick":  last_tick,
    "span":       last_tick - first_tick,
}, index=pd.Index(first_tick, name="first_tick"))

In [ ]:
# Same price, same clock. Only the bar boundaries differ.
fig, (ax_t, ax_c) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))

for ax in (ax_t, ax_c):
    ax.plot(timestamps, price, color="0.5", lw=0.9, marker=".", ms=3)
    ax.set_ylabel("price")

for x in time_mean.index:
    ax_t.axvline(x, color="steelblue", lw=1.0)
ax_t.set_title(f"Time bars (every={EVERY}): equal index width, variable ticks")

for x in first_tick:
    ax_c.axvline(x, color="crimson", lw=1.0)
ax_c.set_title(f"Tick bars (count={COUNT}): equal ticks, variable index width")
ax_c.set_xlabel("index (timestamp)")

plt.tight_layout()

## Empty bars and `fill=`

The hole in the clock is wider than `EVERY`, so some intervals on the grid
contain no ticks at all. `fill=` decides what `resample` emits for them:

- `"skip"` (default): emit no row for an empty bucket.
- `"nan"`: emit an all-NaN row at the empty bucket's label.
- `"carry"`: repeat the previous emitted row's values verbatim.

Only *internal* empty buckets are filled, that is, gaps between two events.
Buckets before the first tick and after the last one are not synthesized on the
eager path.

`"carry"` repeats the whole row uniformly, including count-like columns, so a
genuinely empty bar carries the previous bar's count rather than 0. Use `"nan"`
where that would be wrong.

In [ ]:
modes = ["skip", "nan", "carry"]
frames = {}
for mode in modes:
    bars = resample(price, timestamps, every=EVERY, agg="mean", fill=mode)
    frames[mode] = pd.Series(bars.values, index=bars.index)

# Align the three on the union of their labels to make the gap visible.
pd.DataFrame(frames).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))

ax.plot(timestamps, price, color="0.8", lw=0.9, label="ticks")
for mode, marker in zip(modes, ["o", "s", "^"]):
    s = frames[mode]
    ax.plot(s.index, s.values, marker=marker, ms=5, lw=1.2, label=f'fill="{mode}"')

ax.set_xlabel("index (timestamp)")
ax.set_ylabel("bar mean")
ax.set_title("Empty buckets across the hole under each fill mode")
ax.legend(fontsize=9)
plt.tight_layout()

Under `count=` there is nothing for `fill=` to do: a bar is defined as holding
`N` events, so an empty bar cannot arise. `fill=` is meaningful only with
`every=`.